In [2]:
pip install gtfs-realtime-bindings

Defaulting to user installation because normal site-packages is not writeable

   ---------------------------------------- 0/2 [protobuf]
   ---------------------------------------- 0/2 [protobuf]
   ---------------------------------------- 0/2 [protobuf]
   ---------------------------------------- 0/2 [protobuf]
   ---------------------------------------- 0/2 [protobuf]
   ---------------------------------------- 0/2 [protobuf]
   ---------------------------------------- 0/2 [protobuf]
   ---------------------------------------- 0/2 [protobuf]
   ---------------------------------------- 0/2 [protobuf]
   -------------------- ------------------- 1/2 [gtfs-realtime-bindings]
   ---------------------------------------- 2/2 [gtfs-realtime-bindings]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\RASMITHA\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
from google.transit import gtfs_realtime_pb2
import urllib.request

# MARTA's live bus trip updates URL
feed_url = 'https://gtfs-rt.itsmarta.com/TMGTFSRealTimeWebService/tripupdate/tripupdates.pb'

feed = gtfs_realtime_pb2.FeedMessage()

try:
    response = urllib.request.urlopen(feed_url)
    feed.ParseFromString(response.read())
    
    # Print the first 3 active bus updates to verify it's working
    for entity in feed.entity[:3]:
        print(entity)
        
except Exception as e:
    print(f"Error fetching real-time data: {e}")

id: "11701521"
trip_update {
  trip {
    trip_id: "11701521"
    start_time: "23:10:00"
    start_date: "20260908"
    route_id: "17"
    direction_id: 9
  }
  stop_time_update {
    stop_sequence: 1
    arrival {
      time: 1788923537
      scheduled_time: 1788923400
    }
    departure {
      time: 1788923537
      scheduled_time: 1788923400
    }
    stop_id: "500260"
    stop_time_properties {
      stop_headsign: "\n\006\n\000\022\002en"
      drop_off_type: NONE
    }
  }
  stop_time_update {
    stop_sequence: 2
    arrival {
      time: 1788923578
      scheduled_time: 1788923495
    }
    departure {
      time: 1788923578
      scheduled_time: 1788923495
    }
    stop_id: "214038"
    stop_time_properties {
      stop_headsign: "\n\006\n\000\022\002en"
    }
  }
  stop_time_update {
    stop_sequence: 3
    arrival {
      time: 1788923680
      scheduled_time: 1788923570
    }
    departure {
      time: 1788923680
      scheduled_time: 1788923570
    }
    stop_id: "214

In [4]:
import pandas as pd
from google.transit import gtfs_realtime_pb2
import urllib.request
from datetime import datetime

# MARTA's live bus trip updates URL
feed_url = 'https://gtfs-rt.itsmarta.com/TMGTFSRealTimeWebService/tripupdate/tripupdates.pb'
feed = gtfs_realtime_pb2.FeedMessage()

try:
    response = urllib.request.urlopen(feed_url)
    feed.ParseFromString(response.read())
    
    records = []
    
    # Loop through the feed entities
    for entity in feed.entity:
        if entity.HasField('trip_update'):
            trip = entity.trip_update.trip
            
            # Filter strictly for Buford Highway
            if trip.route_id == "39":
                
                # Check each stop on the trip
                for update in entity.trip_update.stop_time_update:
                    if update.HasField('arrival') and update.arrival.scheduled_time > 0:
                        
                        # Calculate the friction: actual minus scheduled
                        delay_sec = update.arrival.time - update.arrival.scheduled_time
                        
                        records.append({
                            'trip_id': trip.trip_id,
                            'start_time': trip.start_time,
                            'stop_id': update.stop_id,
                            'scheduled_arrival': datetime.fromtimestamp(update.arrival.scheduled_time),
                            'actual_arrival': datetime.fromtimestamp(update.arrival.time),
                            'delay_seconds': delay_sec
                        })
    
    # Convert to Pandas DataFrame for easy analysis
    df = pd.DataFrame(records)
    print(df.head(10))

except Exception as e:
    print(f"Error fetching or processing real-time data: {e}")

    trip_id start_time stop_id   scheduled_arrival      actual_arrival  \
0  11706768   21:30:00  500279 2026-09-08 21:30:00 2026-09-08 23:13:15   
1  11706768   21:30:00  900601 2026-09-08 21:31:42 2026-09-08 23:15:24   
2  11706768   21:30:00  900541 2026-09-08 21:32:48 2026-09-08 23:16:05   
3  11706768   21:30:00  900567 2026-09-08 21:33:49 2026-09-08 23:17:07   
4  11706768   21:30:00  900604 2026-09-08 21:34:54 2026-09-08 23:19:08   
5  11706768   21:30:00  900580 2026-09-08 21:35:47 2026-09-08 23:19:57   
6  11706768   21:30:00  900582 2026-09-08 21:36:54 2026-09-08 23:20:28   
7  11706768   21:30:00  900583 2026-09-08 21:37:32 2026-09-08 23:21:02   
8  11706768   21:30:00  902148 2026-09-08 21:38:23 2026-09-08 23:21:22   
9  11706768   21:30:00  902211 2026-09-08 21:39:05 2026-09-08 23:21:51   

   delay_seconds  
0           6195  
1           6222  
2           6197  
3           6198  
4           6254  
5           6250  
6           6214  
7           6210  
8           61

In [ ]:
import pandas as pd
from google.transit import gtfs_realtime_pb2
import urllib.request
from datetime import datetime
import time
import os

feed_url = 'https://gtfs-rt.itsmarta.com/TMGTFSRealTimeWebService/tripupdate/tripupdates.pb'
csv_filename = 'route39_delay_archive.csv'

print("Starting background data collection. Press Ctrl+C to stop.")

try:
    while True:
        feed = gtfs_realtime_pb2.FeedMessage()
        response = urllib.request.urlopen(feed_url)
        feed.ParseFromString(response.read())
        
        records = []
        for entity in feed.entity:
            if entity.HasField('trip_update') and entity.trip_update.trip.route_id == "39":
                trip = entity.trip_update.trip
                for update in entity.trip_update.stop_time_update:
                    if update.HasField('arrival') and update.arrival.scheduled_time > 0:
                        records.append({
                            'timestamp_pulled': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                            'trip_id': trip.trip_id,
                            'start_time': trip.start_time,
                            'stop_id': update.stop_id,
                            'delay_seconds': update.arrival.time - update.arrival.scheduled_time
                        })
        
        if records:
            df = pd.DataFrame(records)
            # Append to CSV, write header only if the file is new
            df.to_csv(csv_filename, mode='a', header=not os.path.exists(csv_filename), index=False)
            print(f"[{datetime.now().strftime('%H:%M:%S')}] Appended {len(records)} live updates.")
        
        # Pause for 3 minutes before the next API call to avoid rate-limiting
        time.sleep(180)

except KeyboardInterrupt:
    print("\nData collection stopped.")
except Exception as e:
    print(f"Error fetching data: {e}")

Starting background data collection. Press Ctrl+C to stop.
[23:14:38] Appended 228 live updates.
[23:17:38] Appended 228 live updates.
[23:20:39] Appended 228 live updates.
[23:23:39] Appended 213 live updates.
[23:26:40] Appended 200 live updates.
[23:29:40] Appended 190 live updates.
[23:32:40] Appended 189 live updates.
[23:35:41] Appended 174 live updates.
[23:38:41] Appended 174 live updates.
[23:41:42] Appended 164 live updates.
[23:44:42] Appended 196 live updates.
[23:47:42] Appended 196 live updates.
[23:50:43] Appended 192 live updates.
[23:53:43] Appended 176 live updates.
[23:56:43] Appended 176 live updates.
[23:59:44] Appended 174 live updates.
[00:02:44] Appended 153 live updates.
[00:05:44] Appended 141 live updates.
[00:08:45] Appended 141 live updates.
[00:11:45] Appended 141 live updates.


In [1]:
import pandas as pd
from google.transit import gtfs_realtime_pb2
import urllib.request
from datetime import datetime
import time
import os

feed_url = 'https://gtfs-rt.itsmarta.com/TMGTFSRealTimeWebService/tripupdate/tripupdates.pb'
csv_filename = 'route39_delay_archive.csv'

print("Starting background data collection. Press Ctrl+C to stop.")

try:
    while True:
        feed = gtfs_realtime_pb2.FeedMessage()
        response = urllib.request.urlopen(feed_url)
        feed.ParseFromString(response.read())
        
        records = []
        for entity in feed.entity:
            if entity.HasField('trip_update') and entity.trip_update.trip.route_id == "39":
                trip = entity.trip_update.trip
                for update in entity.trip_update.stop_time_update:
                    if update.HasField('arrival') and update.arrival.scheduled_time > 0:
                        records.append({
                            'timestamp_pulled': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                            'trip_id': trip.trip_id,
                            'start_time': trip.start_time,
                            'stop_id': update.stop_id,
                            'delay_seconds': update.arrival.time - update.arrival.scheduled_time
                        })
        
        if records:
            df = pd.DataFrame(records)
            
            df.to_csv(csv_filename, mode='a', header=not os.path.exists(csv_filename), index=False)
            print(f"[{datetime.now().strftime('%H:%M:%S')}] Appended {len(records)} live updates.")
        
        
        time.sleep(180)

except KeyboardInterrupt:
    print("\nData collection stopped.")
except Exception as e:
    print(f"Error fetching data: {e}")

Starting background data collection. Press Ctrl+C to stop.
[11:43:43] Appended 248 live updates.
[11:46:45] Appended 247 live updates.
[11:49:47] Appended 236 live updates.
[11:52:48] Appended 236 live updates.
[11:55:50] Appended 229 live updates.
[11:58:52] Appended 204 live updates.
[12:01:54] Appended 277 live updates.
[12:04:56] Appended 277 live updates.
[12:07:57] Appended 309 live updates.
[12:10:59] Appended 330 live updates.
[12:14:00] Appended 315 live updates.
[12:17:01] Appended 305 live updates.
[12:20:03] Appended 294 live updates.
[12:23:04] Appended 321 live updates.
[12:26:06] Appended 279 live updates.
[12:29:08] Appended 279 live updates.
[12:32:10] Appended 264 live updates.
[12:35:11] Appended 227 live updates.
[12:38:12] Appended 222 live updates.
[12:41:14] Appended 255 live updates.
[12:44:15] Appended 255 live updates.
[12:47:17] Appended 245 live updates.
[12:50:19] Appended 245 live updates.
[12:53:20] Appended 244 live updates.
[14:07:50] Appended 252 live 